In [29]:
def f(z1, z2, b_star):
    """
    Computes the payment as a function of z1, z2, and b_star.

    This uses the simplified version of the expression from the image.
    """

    b2 = 671 + 5 * z2

    # First term:
    # (z1 + 1)/51 * (920 - (671 + 5z1))
    base = (z1 + 1) / 51 * (920 - (671 + 5 * z1))

    # Common second term after cancellation:
    # ((51 - (z1 + 1))/51) * ((z2 - z1)/(51 - (z1 + 1))) * (920 - (671 + 5z2))
    common = (z2 - z1) / 51 * (920 - (671 + 5 * z2))

    if b2 > b_star:
        return base + common
    else:
        denom = 920 - (671 + 5 * z2)

        if denom == 0:
            raise ZeroDivisionError("Denominator 920 - (671 + 5*z2) is zero.")

        factor = ((920 - b_star) / denom) ** 3
        return base + factor * common

In [34]:
import math


def best_at(b_star):
    best_value = float("-inf")
    best_z1 = None
    best_z2 = None

    for z1 in range(49):
        for z2 in range(z1 + 1, 49):
            value = f(z1, z2, b_star)
            if value > best_value:
                best_value = value
                best_z1 = z1
                best_z2 = z2

    return best_z1, best_z2, best_value


def bid_from_z(z):
    return 671 + 5 * z


def step_through(b_star, steps):
    history = []
    current_b_star = b_star

    for step in range(steps):
        z1, z2, value = best_at(current_b_star)
        b1 = bid_from_z(z1)
        b2 = bid_from_z(z2)

        history.append({
            "step": step,
            "b_star": current_b_star,
            "z1": z1,
            "z2": z2,
            "b1": b1,
            "b2": b2,
            "value": value,
        })

        current_b_star = b2

    return history

def step_through_normal(
    b_star,
    steps,
    std=13,
    b_star_low=676,
    b_star_high=919,
    num_samples=60,
):
    history = []
    current_b_star = b_star

    for step in range(steps):
        z1, z2, value = best_at_normal(
            current_b_star,
            std,
            b_star_low=b_star_low,
            b_star_high=b_star_high,
            num_samples=num_samples,
        )
        b1 = bid_from_z(z1)
        b2 = bid_from_z(z2)

        history.append({
            "step": step,
            "b_star": current_b_star,
            "z1": z1,
            "z2": z2,
            "b1": b1,
            "b2": b2,
            "value": value,
        })

        current_b_star = b2

    return history


def pnl_assuming_z2_above_b_star(z1, z2):
    base = (z1 + 1) / 51 * (920 - bid_from_z(z1))
    common = (z2 - z1) / 51 * (920 - bid_from_z(z2))
    return base + common


def best_z2_for_z1(z1):
    best_value = float("-inf")
    best_z2 = None

    for z2 in range(z1 + 1, 49):
        value = pnl_assuming_z2_above_b_star(z1, z2)
        if value > best_value:
            best_value = value
            best_z2 = z2

    return best_z2, best_value


def normal_weight(x, mean, std):
    if std <= 0:
        return 1.0 if x == mean else 0.0
    return math.exp(-0.5 * ((x - mean) / std) ** 2)


def expected_f_under_normal(
    z1,
    z2,
    expected_b_star,
    std,
    b_star_low=676,
    b_star_high=919,
    num_samples=60,
):
    if std <= 0:
        return f(z1, z2, expected_b_star)

    if num_samples <= 1:
        b_star_samples = [expected_b_star]
    else:
        step = (b_star_high - b_star_low) / (num_samples - 1)
        b_star_samples = [b_star_low + i * step for i in range(num_samples)]

    total_weight = 0
    total_value = 0

    for b_star in b_star_samples:
        weight = normal_weight(b_star, expected_b_star, std)
        total_weight += weight
        total_value += weight * f(z1, z2, b_star)

    return total_value / total_weight


def best_at_normal(
    expected_b_star,
    std = 13,
    b_star_low=676,
    b_star_high=919,
    num_samples=60,
):
    best_value = float("-inf")
    best_z1 = None
    best_z2 = None

    for z1 in range(49):
        for z2 in range(z1 + 1, 49):
            value = expected_f_under_normal(
                z1,
                z2,
                expected_b_star,
                std,
                b_star_low=b_star_low,
                b_star_high=b_star_high,
                num_samples=num_samples,
            )
            if value > best_value:
                best_value = value
                best_z1 = z1
                best_z2 = z2

    return best_z1, best_z2, best_value

best_z2_for_z1(24) #is 37

step_through_normal(bid_from_z(37), steps=3)

[{'step': 0,
  'b_star': 856,
  'z1': 19,
  'z2': 39,
  'b1': 766,
  'b2': 866,
  'value': 80.04933930292546},
 {'step': 1,
  'b_star': 866,
  'z1': 20,
  'z2': 41,
  'b1': 771,
  'b2': 876,
  'value': 77.92433869569051},
 {'step': 2,
  'b_star': 876,
  'z1': 21,
  'z2': 42,
  'b1': 776,
  'b2': 881,
  'value': 75.5571287760113}]